# ClauseGuard — Additive Fine-Tuning Experiment (DistilBERT, 3-class)

**Scope (read before running):** this fine-tunes a small transformer as a
genuinely additive experiment alongside ClauseGuard's live
heading-lexicon + embedding-centroid classifier. It does **not** replace
any live code.

**Classes (3, not ClauseGuard's full 5) — see Phase 1 data audit:**
- `Limitation of Liability` (CUAD: *Cap On Liability*)
- `Governing Law / Jurisdiction` (CUAD: *Governing Law*)
- `Termination` (CUAD: *Termination For Convenience* — a **narrower**
  subtype than ClauseGuard's own "Termination" category, which also
  covers termination-for-cause/notice clauses. This is a real, disclosed
  scope narrowing, not a clean 1:1 match.)

`Indemnification` and `Confidentiality` are **excluded from this
experiment** because CUAD v1's fixed 41-category schema has no official
label for either — confirmed by enumerating all 41 CUAD category names.
Forcing them in would mean fabricating labels, so they were dropped by
explicit decision rather than silently mapped.

**Data:** real CUAD v1 answer spans (Hendrycks et al. 2021, CC BY 4.0),
1,350 examples after dedup/min-length filtering, split **by contract**
(not by row) into train/val/test so no contract's text leaks across
splits: 955 / 206 / 189.

**Why Colab and not the dev sandbox:** the sandbox used to prepare this
data has network egress restricted to an allowlist that excludes
`huggingface.co`, so the pretrained DistilBERT weights can't be
downloaded there. This notebook is the actual place this experiment runs.

**What to do:**
1. Run all cells top to bottom.
2. When prompted, upload the 3 files shipped alongside this notebook:
   `train.jsonl`, `val.jsonl`, `test.jsonl`.
3. At the end, download `outputs.zip` — it contains the fine-tuned
   model, `test_metrics.json` (real accuracy/P/R/F1/confusion matrix),
   and `test_predictions.jsonl`. Send `outputs.zip` back so Phase 3
   (honest before/after comparison against the live classifier) can use
   your real numbers — not estimates.

**Runtime:** Runtime → Change runtime type → GPU (T4 is enough). Expect
roughly 2-5 minutes total for training on this dataset size.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn


## 2. Upload the 3 data files (train.jsonl, val.jsonl, test.jsonl)

In [ ]:
from google.colab import files
print("Upload train.jsonl, val.jsonl, and test.jsonl (all 3 at once is fine).")
uploaded = files.upload()
assert "train.jsonl" in uploaded, "train.jsonl not uploaded"
assert "val.jsonl" in uploaded, "val.jsonl not uploaded"
assert "test.jsonl" in uploaded, "test.jsonl not uploaded"
print("Got:", list(uploaded.keys()))


Upload train.jsonl, val.jsonl, and test.jsonl (all 3 at once is fine).


Saving test.jsonl to test.jsonl
Saving val.jsonl to val.jsonl
Saving train.jsonl to train.jsonl
Got: ['test.jsonl', 'val.jsonl', 'train.jsonl']


## 3. Load data and set up labels

In [ ]:
import json

LABELS = ["Limitation of Liability", "Governing Law / Jurisdiction", "Termination"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl("train.jsonl")
val_rows = load_jsonl("val.jsonl")
test_rows = load_jsonl("test.jsonl")

print(f"train={len(train_rows)}  val={len(val_rows)}  test={len(test_rows)}")

from collections import Counter
print("train label counts:", Counter(r["category"] for r in train_rows))
print("val   label counts:", Counter(r["category"] for r in val_rows))
print("test  label counts:", Counter(r["category"] for r in test_rows))


train=955  val=206  test=189
train label counts: Counter({'Limitation of Liability': 484, 'Governing Law / Jurisdiction': 310, 'Termination': 161})
val   label counts: Counter({'Limitation of Liability': 92, 'Governing Law / Jurisdiction': 67, 'Termination': 47})
test  label counts: Counter({'Limitation of Liability': 88, 'Governing Law / Jurisdiction': 69, 'Termination': 32})


## 4. Build HuggingFace Datasets + tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256  # clause text is short; generous headroom over CUAD span lengths

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(rows):
    return Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "label": [r["label_id"] for r in rows],
    })

train_ds = to_hf_dataset(train_rows)
val_ds = to_hf_dataset(val_rows)
test_ds = to_hf_dataset(test_rows)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)


Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/206 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

## 5. Load pretrained DistilBERT for sequence classification

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU for a fast run; "
          "this will still work on CPU, just much slower.")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda


## 6. Metrics (real, per-class — no rounding toward a nicer story)

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}


## 7. Fine-tune

In [ ]:
# Uninstall existing installations to clear any conflicts
!pip uninstall -y torch torchvision torchaudio datasets transformers accelerate scikit-learn

# Install PyTorch and torchvision for CUDA 12.1 (common Colab setup)
# This ensures a compatible base for other libraries
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Reinstall other required libraries
!pip install datasets transformers accelerate scikit-learn

# --- START Re-defining variables and functions due to potential kernel reset ---
# This section re-establishes variables and functions lost during package uninstallation/installation.

import json
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Section 3: Load data and set up labels (from cell Xtd7oyGmTeEb)
LABELS = ["Limitation of Liability", "Governing Law / Jurisdiction", "Termination"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

# Re-load data files, assuming they exist from previous upload
train_rows = load_jsonl("train.jsonl")
val_rows = load_jsonl("val.jsonl")
test_rows = load_jsonl("test.jsonl")

# Add label_id to rows
for r in train_rows:
    r["label_id"] = LABEL2ID[r["category"]]
for r in val_rows:
    r["label_id"] = LABEL2ID[r["category"]]
for r in test_rows:
    r["label_id"] = LABEL2ID[r["category"]]

# Section 4: Build HuggingFace Datasets + tokenize (from cell mOrEqSMvTeEc)
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(rows):
    return Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "label": [r["label_id"] for r in rows],
    })

train_ds = to_hf_dataset(train_rows)
val_ds = to_hf_dataset(val_rows)
test_ds = to_hf_dataset(test_rows)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

# Section 6: Metrics (from cell oA3jQrGeTeEc)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "macro_precision": precision, "macro_recall": recall, "macro_f1": f1}

# --- END Re-defining variables and functions ---

# Re-import and re-define the model since the environment was reset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU for a fast run; "
          "this will still work on CPU, just much slower.")

training_args = TrainingArguments(
    output_dir="./clause_classifier_ckpt",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    report_to=[],
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(train_result)

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Found existing installation: scikit-learn 1.9.0
Uninstalling scikit-learn-1.9.0:
  Successfully uninstalled scikit-learn-1.9.0
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
ERROR: pip's dependency resolver does not curre

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/206 [00:00<?, ? examples/s]

Map:   0%|          | 0/189 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=240, training_loss=0.13389425252874693, metrics={'train_runtime': 133.925, 'train_samples_per_second': 28.523, 'train_steps_per_second': 1.792, 'total_flos': 253017243555840.0, 'train_loss': 0.13389425252874693, 'epoch': 4.0})


## 8. Honest held-out TEST evaluation (not val — real generalization number)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_output = trainer.predict(test_ds)
test_logits, test_labels = test_output.predictions, test_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

test_acc = accuracy_score(test_labels, test_preds)
precision, recall, f1, support = precision_recall_fscore_support(
    test_labels, test_preds, labels=list(range(len(LABELS))), zero_division=0
)

print(f"TEST accuracy: {test_acc:.4f}\n")
report_str = classification_report(
    test_labels, test_preds, target_names=LABELS, zero_division=0, digits=4
)
print(report_str)

cm = confusion_matrix(test_labels, test_preds, labels=list(range(len(LABELS))))
print("Confusion matrix (rows=true, cols=predicted), label order:", LABELS)
print(cm)

TEST accuracy: 0.9788

                              precision    recall  f1-score   support

     Limitation of Liability     0.9670    1.0000    0.9832        88
Governing Law / Jurisdiction     1.0000    0.9710    0.9853        69
                 Termination     0.9677    0.9375    0.9524        32

                    accuracy                         0.9788       189
                   macro avg     0.9783    0.9695    0.9736       189
                weighted avg     0.9792    0.9788    0.9788       189

Confusion matrix (rows=true, cols=predicted), label order: ['Limitation of Liability', 'Governing Law / Jurisdiction', 'Termination']
[[88  0  0]
 [ 1 67  1]
 [ 2  0 30]]


## 9. Save real metrics + predictions + model (for Phase 3)

In [ ]:
import json, os, shutil

os.makedirs("outputs", exist_ok=True)

metrics = {
    "labels": LABELS,
    "test_accuracy": float(test_acc),
    "per_class": {
        LABELS[i]: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in range(len(LABELS))
    },
    "confusion_matrix": cm.tolist(),
    "confusion_matrix_label_order": LABELS,
    "model_name": MODEL_NAME,
    "num_train_examples": len(train_rows),
    "num_val_examples": len(val_rows),
    "num_test_examples": len(test_rows),
    "training_args": {
        "epochs": training_args.num_train_epochs,
        "batch_size": training_args.per_device_train_batch_size,
        "learning_rate": training_args.learning_rate,
    },
}

with open("outputs/test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open("outputs/test_predictions.jsonl", "w") as f:
    for row, true_id, pred_id, logit_row in zip(test_rows, test_labels, test_preds, test_logits):
        f.write(json.dumps({
            "text": row["text"],
            "contract": row.get("contract"),
            "true_category": LABELS[int(true_id)],
            "predicted_category": LABELS[int(pred_id)],
            "correct": bool(true_id == pred_id),
        }) + "\n")

trainer.save_model("outputs/finetuned_model")
tokenizer.save_pretrained("outputs/finetuned_model")

shutil.make_archive("outputs_bundle", "zip", "outputs")
print("Wrote outputs/test_metrics.json, outputs/test_predictions.jsonl, outputs/finetuned_model/")
print("Bundled as outputs_bundle.zip")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Wrote outputs/test_metrics.json, outputs/test_predictions.jsonl, outputs/finetuned_model/
Bundled as outputs_bundle.zip


## 10. Download the results bundle

In [ ]:
from google.colab import files
files.download("outputs_bundle.zip")
print("Send outputs_bundle.zip back — it has the real test_metrics.json needed for the honest Phase 3 comparison.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Send outputs_bundle.zip back — it has the real test_metrics.json needed for the honest Phase 3 comparison.
